In [ ]:
import os

SHARED_FOLDER = ("/content/drive/MyDrive/data_handling_and_machine_learning/week_05/outputs/")

json_files = [
    f for f in os.listdir(SHARED_FOLDER)
    if f.lower().endswith(".json")
]

print("JSON files found:")
for f in sorted(json_files):
    print("-", f)

JSON files found:
- dinal_ibm_processed_chunks.json
- gam_nvda_processed_chunks.json
- olga_processed_chunks.json
- sandip_UPS_Uber_chunks.json


# Merge all 4 chunk datasets

In [ ]:
import json
import os

SHARED_FOLDER = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/merged_chunks_RAG/")

# Add your team's JSON files here
JSON_FILES = [
    "dinal_ibm_processed_chunks.json",
    "gam_nvda_processed_chunks.json",
    "olga_processed_chunks.json",
    "sandip_UPS_Uber_chunks.json",
]

MERGED_JSON_PATH = os.path.join(SHARED_FOLDER, "InvestIQ_all_RAG_chunks.json")

merged_chunks = []

for filename in JSON_FILES:
    file_path = os.path.join(SHARED_FOLDER, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"{filename}: {len(data)} chunks")

    merged_chunks.extend(data)

with open(MERGED_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        merged_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print()
print("Merge completed.")
print("Total chunks:", len(merged_chunks))
print("Saved to:", MERGED_JSON_PATH)

dinal_ibm_processed_chunks.json: 196 chunks
gam_nvda_processed_chunks.json: 116 chunks
olga_processed_chunks.json: 405 chunks
sandip_UPS_Uber_chunks.json: 373 chunks

Merge completed.
Total chunks: 1090
Saved to: /content/drive/MyDrive/data_handling_and_machine_learning/week_05/outputs/InvestIQ_all_RAG_chunks.json


# Validate the merged dataset

In [ ]:
import json

with open(MERGED_JSON_PATH, "r", encoding="utf-8") as f:
    merged_data = json.load(f)

print("Total chunks:", len(merged_data))

# Check structure
missing_text = 0
missing_metadata = 0
missing_chunk_id = 0

chunk_ids = []

for item in merged_data:
    if "text" not in item:
        missing_text += 1

    if "metadata" not in item:
        missing_metadata += 1
        continue

    if "chunk_id" not in item["metadata"]:
        missing_chunk_id += 1
    else:
        chunk_ids.append(item["metadata"]["chunk_id"])

print("Missing text:", missing_text)
print("Missing metadata:", missing_metadata)
print("Missing chunk_id:", missing_chunk_id)

# Check duplicate chunk IDs
duplicate_ids = len(chunk_ids) - len(set(chunk_ids))

print("Duplicate chunk IDs:", duplicate_ids)

Total chunks: 1090
Missing text: 0
Missing metadata: 0
Missing chunk_id: 0
Duplicate chunk IDs: 0


# Inspect the existing shared collection

In [ ]:
!pip install -q -U chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
import chromadb

CHROMA_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/chroma_db")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name="investiq_rag")

print("Collection:", collection.name)
print("Current chunks:", collection.count())

existing_data = collection.get(
    include=["metadatas"]
)

companies = {}

for metadata in existing_data["metadatas"]:
    company = metadata.get("company", "Unknown")
    companies[company] = companies.get(company, 0) + 1

print("\nExisting companies:")
for company, count in sorted(companies.items()):
    print(f"- {company}: {count}")

Collection: investiq_rag
Current chunks: 0

Existing companies:


In [ ]:
from tqdm.auto import tqdm

BATCH_SIZE = 100

for start in tqdm(
    range(0, len(merged_data), BATCH_SIZE),
    desc="Adding chunks"
):
    batch = merged_data[start:start + BATCH_SIZE]

    documents = [item["text"] for item in batch]

    metadatas = [item["metadata"] for item in batch]

    ids = [item["metadata"]["chunk_id"] for item in batch]

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )

print("\nImport completed.")
print("Collection count:", collection.count())

Adding chunks:   0%|          | 0/11 [00:00<?, ?it/s]


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 0.00/79.3M [00:00<?, ?iB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   1%|          | 986k/79.3M [00:00<00:08, 9.47MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  14%|█▍        | 11.2M/79.3M [00:00<00:01, 65.4MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  28%|██▊       | 22.3M/79.3M [00:00<00:00, 88.3MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  42%|████▏     | 33.6M/79.3M [00:00<00:00, 100MiB/s] 
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  55%|█████▌    | 43.8M/79.3M [00:00<00:00, 102MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  69%|██████▉   | 54.6M/79.3M [00:00<00:00, 106MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  82%|████████▏ | 65.2M/79.3M [00:00<00:00, 108MiB/s]
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M


Import completed.
Collection count: 1090


In [ ]:
print("Final RAG collection:", collection.name)
print("Total chunks:", collection.count())

sample = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

print("\nSample chunks:")
for i, (document, metadata) in enumerate(
    zip(sample["documents"], sample["metadatas"]),
    start=1
):
    print("=" * 80)
    print(f"Sample {i}")
    print(f"Company: {metadata.get('company')}")
    print(f"Section: {metadata.get('section')}")
    print(f"Chunk ID: {metadata.get('chunk_id')}")
    print(f"Text preview: {document[:300]}...")

Final RAG collection: investiq_rag
Total chunks: 1090

Sample chunks:
Sample 1
Company: International Business Machines Corporation
Section: Business
Chunk ID: IBM_Business_001
Text preview: Item 1. Business:
International Business Machines Corporation (IBM or the company) was incorporated in the State of New York on June 16, 1911, as the Computing-Tabulating-Recording Co. (C-T-R), a consolidation of the Computing Scale Co. of America, the Tabulating Machine Co. and The International Ti...
Sample 2
Company: International Business Machines Corporation
Section: Business
Chunk ID: IBM_Business_002
Text preview: Our breadth and depth of expertise enables us to take different technologies and bring them together to solve the most pressing business issues of our clients. We differentiate from other providers by bringing together incumbency with enterprises, deep expertise in technology, industries and busines...
Sample 3
Company: International Business Machines Corporation
Section: Business
